**Fig. 4D and Video S4: whole-tissue run on the scale condensate stage with the DSM pair**

Needs a GPU.

In [ ]:
# Working directory: data/fig4/ under the repository root, where the Fig. 4 deposit folder is downloaded
# (domain-specialized_finetuned_bootstrap/ = the DSM model pair, one folder per sample with its
# input volume and the final mask, wound_finetuning/).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig4")
print("working directory:", os.getcwd())


# EpiCycle ScaledSegmentation Pipeline

Run 3D cell segmentation, reconstruction, and assembly on tiled image data.

**Usage:** Fill in the paths and parameters below, then Run All.

## 1. Paths & Input

In [ ]:
# ===== PATHS (edit these) =====

# Input Z-stack TIFF (the original microscope file name is in the README) — set this to tile from scratch, or leave None if already tiled
INPUT_TIF = "scale_condensate/input.tif"   # Set to None if tiles already exist

# Working directory (tiles + outputs go here)
# If INPUT_TIF is set, tiles will be created at WORK_DIR/tiles/
# If INPUT_TIF is None, WORK_DIR should contain existing tile subdirectories
WORK_DIR = "scale_condensate"

# CellPose model paths (.pth files)
LATERAL_MODEL = "domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootstrap4Finetuned_lateral_500epoch_scratch_lenient_rescale_True"
AXIAL_MODEL = "domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootrap4Finetuned_axial_500epoch_scratch_lenient_rescale_True"

# Output directory name (created inside WORK_DIR)
OUTPUT_NAME = "ScaleCondensate"

## 2. Resource Configuration

In [ ]:
# ===== RESOURCES (auto-detected from the job's allocation if None) =====

N_GPUS = None            # None = auto-detect from SLURM/CUDA
N_CPU_PER_WORKER = None  # None = auto (total_cores / n_gpus in parallel mode)

## 3. Stage Selection & Tiling Parameters

In [ ]:
# Which stages to run (set False to skip)
RUN_TILING = INPUT_TIF is not None   # Auto: tile if input TIFF provided
# RUN_TILING = False
RUN_SEGMENTATION = True
RUN_RECONSTRUCTION = True
RUN_ASSEMBLY = True 
RUN_GLOBAL_RECONSTRUCTION = True

# ===== TILING (Stage 0) =====
PATCH_SIZE = 1024           # Target XY patch size in pixels
# OVERLAP is shared with assembly (set in Section 6)
OVERLAP = 0.25

Z_THRESHOLD_METHOD = "hard"  # "hard", "percentile", or "default"
Z_THRESHOLD_VALUE = 0.07     # Intensity threshold for Z-range detection
CHANNEL_INDEX = 1            # Which channel for Z-range analysis
AUTO_CORRECT_PATCH = False   # Auto-find valid patch size if exact tiling fails

## 4. Segmentation Parameters

In [ ]:
# ===== SEGMENTATION (Stage 1) =====
# Recorded for reference: the stage is built without segment_params below, so the engine
# defaults were used (channels [2, 1], cell probability 0, flow threshold 0.2). The flow
# threshold acts on the 2D masks only; the 3D reconstruction uses the probability and flow maps.

# GPU allocation strategy: "auto", "fixed" (3-GPU triplets), "round_robin"
GPU_STRATEGY = "auto"

# CellPose parameters
FLOW_THRESHOLD = 0.4
CELLPROB_THRESHOLD = 0.0
CHANNELS = [2, 1]      # [signal, nucleus] — adjust for your data
BATCH_SIZE = 1024

## 5. Reconstruction Parameters

In [ ]:
# ===== RECONSTRUCTION (Stage 2) =====

# Per-tile direct reconstruction: the engine's u-Segment3D defaults, with gradient renormalisation
# off as in the engine version the runs used (EpiCycle v0.1.2 defaults to on).
RECON_RENORM_GRADIENT = False

# "sequential" (one tile at a time, all cores) or "parallel" (multiple tiles)
RECON_MODE = "parallel"
# RECON_MODE = "sequential"

# Max concurrent tiles in parallel mode (None = n_gpus)
# Rule of thumb: 1900 TB node / ~450 GB per 1920x1920 tile ≈ 4 max
RECON_MAX_WORKERS = None

# Post-processing
SAVE_PROB3D = True
SAVE_FLOW3D = True
SAVE_BINARY = True
BINARY_ERODE = True
BINARY_KERNEL_SIZE = 3
BINARY_BORDER_MARGIN = 2
BINARY_FILL_HOLES = True

# Delete stage1 intermediates after reconstruction to save disk
CLEANUP_INTERMEDIATES = False

## 6. Assembly Parameters

In [ ]:
# ===== ASSEMBLY (Stage 3) =====

# Full volume dimensions in pixels (H, W)
FULL_SIZE = (1792, 1792)

# Tile size and overlap (must match how tiles were cut)
TILE_SIZE = PATCH_SIZE

# Number of Z slices (None = auto-detect from first tile)
DEPTH = None

# Which volumes to stitch
STITCH_MASK = True
STITCH_PROB = True
STITCH_BINARY = True

# Overlap blending for probabilities: "average" or "max"
PROB_STITCH_MODE = "max"

## 7. Global Indirect Reconstruction Parameters

In [ ]:
# ===== GLOBAL INDIRECT RECONSTRUCTION (Stage 4) =====

DOWNSAMPLE_FACTOR = 2          # Downsample stitched volume by this factor
GLOBAL_MIN_SIZE = 1000         # Remove objects smaller than this (voxels, after downsampling)
USE_PRECOMPUTED_BINARY = True  # Use stitched binary mask as foreground constraint

# Gradient descent parameters
# GLOBAL_N_ITER = 30            # Gradient descent iterations (more = finer, slower)
GLOBAL_N_ITER = 300            # Gradient descent iterations (more = finer, slower)
GLOBAL_MOMENTA = 0.9          # Momentum for gradient descent
# GLOBAL_GRADIENT_DECAY = 0.001    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.01    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.1    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.05    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
GLOBAL_GRADIENT_DECAY = 0.25    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower

GLOBAL_RENORM_GRADIENT = True  # Renormalize gradient at each step

# Indirect method parameters
# GLOBAL_DTFORM_METHOD = "edt"   # Distance transform method edt is simple but there maybe artifacts in the seems
GLOBAL_DTFORM_METHOD = "edt"   # Distance transform method edt is simple but there maybe artifacts in the seems
GLOBAL_SMOOTH_SKEL_SIGMA = 1   # Skeleton smoothing sigma
GLOBAL_N_CPU = 64              # CPU threads for multiprocess watershed
GLOBAL_TILE_SHAPE = None       # Tile shape for watershed (None = auto)

## 8. Setup & Pre-flight Check

In [ ]:
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

# Resolve GPU count
from packages.epicycle_scaledseg.utils.resources import get_allocated_gpus
n_gpus = N_GPUS or get_allocated_gpus()

work_dir = Path(WORK_DIR)
work_dir.mkdir(parents=True, exist_ok=True)

# Output lives inside the working directory
output_dir = work_dir / OUTPUT_NAME
output_dir.mkdir(exist_ok=True)
log_dir = output_dir / "logs"
log_dir.mkdir(exist_ok=True)

# Tiles directory
tiles_dir = work_dir / "tiles"

if RUN_TILING:
    print(f"Input TIFF: {INPUT_TIF}")
    print(f"Tiles will be created in: {tiles_dir}")
    print(f"Patch size: {PATCH_SIZE}, overlap: {OVERLAP}")
else:
    # Discover existing tile directories
    tile_dirs = sorted([
        str(p) for p in work_dir.iterdir()
        if p.is_dir() and not p.name.startswith(("stage1", "stage2", "stage3", "stage4", "logs", ".", "epicycle"))
        and p.name != OUTPUT_NAME
    ])
    print(f"Found {len(tile_dirs)} existing tile directories in {work_dir}")
    for td in tile_dirs[:5]:
        print(f"  {Path(td).name}")
    if len(tile_dirs) > 5:
        print(f"  ... and {len(tile_dirs) - 5} more")

print(f"\nGPUs: {n_gpus}")
print(f"Output: {output_dir}")

Input TIFF: <run>/WT_Developing_Fish/8-15-25_1791-2012_cdh1GFP_actb-h2b-mSc_SL8._scale condensat - Stitched - Denoised.nd2-PROCESSED-ISO-1792x1792.tif
Tiles will be created in: <run>/WT_Developing_Fish/ScaleCondensate/tiles
Patch size: 1024, overlap: 0.25

GPUs: 3
Output: <run>/WT_Developing_Fish/ScaleCondensate/ScaleCondensate


## 9. Build & Run Pipeline

In [ ]:
from packages.epicycle_scaledseg.core.pipeline import EpiCyclePipeline

pipeline = EpiCyclePipeline(
    work_dir=str(work_dir),
    n_gpus=n_gpus,
    log_dir=str(log_dir),
)

# --- Stage 0: Tiling ---
if RUN_TILING:
    from packages.epicycle_scaledseg.core.tiling import TilingStage
    pipeline.add_stage(TilingStage(
        input_tif=INPUT_TIF,
        output_dir=str(tiles_dir),
        patch_size=PATCH_SIZE,
        overlap=OVERLAP,
        z_threshold_method=Z_THRESHOLD_METHOD,
        threshold_value=Z_THRESHOLD_VALUE,
        channel_index=CHANNEL_INDEX,
        auto_correct=AUTO_CORRECT_PATCH,
    ))
    # Tiling creates tile_dirs; use placeholder for now
    tile_dirs = [str(work_dir)]

# --- Stage 1: Segmentation ---
if RUN_SEGMENTATION:
    from packages.epicycle_scaledseg.core.segmentation import SegmentationStage
    pipeline.add_stage(SegmentationStage(
        lateral_model=LATERAL_MODEL,
        axial_model=AXIAL_MODEL,
        out_root=str(output_dir / "stage1_segmentation"),
        gpu_strategy=GPU_STRATEGY,
    ))

# --- Stage 2: Reconstruction ---
if RUN_RECONSTRUCTION:
    from packages.epicycle_scaledseg.core.reconstruction import ReconstructionStage
    from packages.epicycle_scaledseg.config.defaults import get_default_reconstruct_params
    reconstruct_params = get_default_reconstruct_params()
    reconstruct_params['gradient_descent']['renorm_gradient'] = RECON_RENORM_GRADIENT
    pipeline.add_stage(ReconstructionStage(
        seg_root=str(output_dir / "stage1_segmentation"),
        out_root=str(output_dir / "stage2_reconstruction"),
        mode=RECON_MODE,
        reconstruct_params=reconstruct_params,
        max_workers=RECON_MAX_WORKERS,
        n_cpu_per_worker=N_CPU_PER_WORKER,
        save_prob3d=SAVE_PROB3D,
        save_flow3d=SAVE_FLOW3D,
        save_binary=SAVE_BINARY,
        binary_erode=BINARY_ERODE,
        binary_kernel_size=BINARY_KERNEL_SIZE,
        binary_border_margin=BINARY_BORDER_MARGIN,
        binary_fill_holes=BINARY_FILL_HOLES,
        cleanup_intermediates=CLEANUP_INTERMEDIATES,
    ))

# --- Stage 3: Assembly ---
if RUN_ASSEMBLY:
    from packages.epicycle_scaledseg.core.assembly import AssemblyStage
    pipeline.add_stage(AssemblyStage(
        recon_root=str(output_dir / "stage2_reconstruction"),
        output_dir=str(output_dir / "stage3_assembly"),
        full_size=FULL_SIZE,
        tile_size=TILE_SIZE,
        overlap=OVERLAP,
        depth=DEPTH,
        stitch_mask=STITCH_MASK,
        stitch_prob=STITCH_PROB and SAVE_PROB3D,
        stitch_binary=STITCH_BINARY and SAVE_BINARY,
        prob_stitch_mode=PROB_STITCH_MODE,
    ))

# --- Stage 4: Global Indirect Reconstruction ---
if RUN_GLOBAL_RECONSTRUCTION:
    from packages.epicycle_scaledseg.core.global_reconstruction import GlobalReconstructionStage
    from packages.epicycle_scaledseg.config.defaults import get_default_indirect_reconstruct_params

    indirect_params = get_default_indirect_reconstruct_params(n_cpu=GLOBAL_N_CPU)
    indirect_params['gradient_descent']['n_iter'] = GLOBAL_N_ITER
    indirect_params['gradient_descent']['momenta'] = GLOBAL_MOMENTA
    indirect_params['gradient_descent']['gradient_decay'] = GLOBAL_GRADIENT_DECAY
    indirect_params['gradient_descent']['renorm_gradient'] = GLOBAL_RENORM_GRADIENT
    indirect_params['indirect_method']['dtform_method'] = GLOBAL_DTFORM_METHOD
    indirect_params['indirect_method']['smooth_skel_sigma'] = GLOBAL_SMOOTH_SKEL_SIGMA
    if GLOBAL_TILE_SHAPE is not None:
        indirect_params['gradient_descent']['tile_shape'] = GLOBAL_TILE_SHAPE

    pipeline.add_stage(GlobalReconstructionStage(
        assembly_dir=str(output_dir / "stage3_assembly"),
        output_dir=str(output_dir / "stage4_global"),
        downsample_factor=DOWNSAMPLE_FACTOR,
        min_size=GLOBAL_MIN_SIZE,
        use_precomputed_binary=USE_PRECOMPUTED_BINARY,
        indirect_params=indirect_params,
        n_cpu=GLOBAL_N_CPU,
        tile_shape=GLOBAL_TILE_SHAPE,
    ))

print(f"Pipeline: {len(pipeline.stages)} stages")
for s in pipeline.stages:
    print(f"  - {s.name}")
print(f"Tile dirs: {len(tile_dirs)}")
print(f"Output: {output_dir}")

# ===== RUN =====
result = pipeline.run(tile_dirs)
print()
print(result.summary())

2026-02-04 14:06:09,552 epicycle.pipeline INFO Pipeline starting: 5 stage(s), 1 tile(s), 3 GPU(s)
2026-02-04 14:06:09,553 epicycle.pipeline INFO Run parameters saved to data/WT_Developing_Fish/ScaleCondensate/ScaleCondensate/logs/run_params.json
2026-02-04 14:06:09,553 epicycle.pipeline INFO === Stage: tiling ===
2026-02-04 14:06:09,554 epicycle.tiling INFO Stage 'tiling' starting with 1 tile(s)
2026-02-04 14:06:09,555 epicycle.tiling INFO Parameters: {"input_tif": "data/WT_Developing_Fish/8-15-25_1791-2012_cdh1GFP_actb-h2b-mSc_SL8._scale condensat - Stitched - Denoised.nd2-PROCESSED-ISO-1792x1792.tif", "patch_size": 1024, "overlap": 0.25, "z_threshold_method": "hard", "threshold_value": 0.07, "channel_index": 1, "auto_correct": false}
2026-02-04 14:06:09,556 epicycle.core.tiling INFO Tiling data/WT_Developing_Fish/8-15-25_1791-2012_cdh1GFP_actb-h2b-mSc_SL8._scale condensat - Stitched - Denoised.nd2-PROCESSED-ISO-1792x1792.tif -> data/WT_Developing_Fish/ScaleCondensate/tiles (patch=102

Pipeline: 5 stages
  - tiling
  - segmentation
  - reconstruction
  - assembly
  - global_reconstruction
Tile dirs: 1
Output: <run>/WT_Developing_Fish/ScaleCondensate/ScaleCondensate


2026-02-04 14:06:16,011 epicycle.utils.tiling INFO Z-stack shape: Z=1031, C=2, H=1792, W=1792, dtype=uint16
2026-02-04 14:06:16,012 epicycle.utils.tiling INFO Tiling: patch_size=1024, step=768, overlap=0.25
2026-02-04 14:06:16,012 epicycle.utils.tiling INFO Grid: 2 rows x 2 cols = 4 tiles
Saving tiles:   0%|                                                                                                                           | 0/4 [00:00<?, ?tile/s]<env>/site-packages/tifffile/tifffile.py:3801: UserWarning: <tifffile.TiffWriter 'tile_0_0_segmentrange_0_779.tif'> truncating ImageJ file
  warnings.warn(
Saving tiles:  25%|████████████████████████████▊                                                                                      | 1/4 [00:02<00:06,  2.13s/tile]<env>/site-packages/tifffile/tifffile.py:3801: UserWarning: <tifffile.TiffWriter 'tile_0_1_segmen…ange_212_937.tif'> truncating ImageJ file
  warnings.warn(
Saving tiles:  50%|██████████████████████████████████████████████

Automatric binary thresholding:  0.6
[ 465 1216 1216] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  180
start dynamics
Tile id 28 0 0
Tile id 18 0 0
Tile id 26 0 0
Tile id 22 0 0
Tile id 24 0 0
Tile id 31 0 0
Tile id 29 0 0
Tile id 35 0 0
Tile id 19 0 0
Tile id 30 0 0
Tile id 13 0 0
Tile id 34 0 0
Tile id 20 0 0
Tile id 27 0 0
Tile id 25 0 0
Tile id 32 0 0
Tile id 21 0 0
Tile id 33 0 0
Tile id 57 0 0
Tile id 62 0 0
Tile id 61 0 0
Tile id 56 404 404
Tile id 65 0 0
Tile id 64 0 0
Tile id 63 0 0
Tile id 66 0 0
Tile id 70 0 0
Tile id 69 0 0
Tile id 60 1199 1199
Tile id 67 0 0
Tile id 68 0 0
Tile id 71 0 0
Tile id 12 9642 9642
Tile id 5 20016 20016
Tile id 23 33826 33826
Tile id 14 14804 14804
Tile id 11 25837 25837
Tile id 10 44785 44785
Tile id 16 30183 30183
Tile id 15 63529 63529
Tile id 55 31842 31842
Tile id 17 66703 66703
Tile id 51 76204 76204
Tile id 72 11551 11551
Tile id 73 23636 23636
Tile id 58 74109 74109
Tile id 9 96838 96838
Tile id 54 101791 101791
Tile id 5

Eroding (GPU): 100%|██████████| 832/832 [00:03<00:00, 241.34inst/s] 


Automatric binary thresholding:  0.6
[ 444 1216 1216] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  180
start dynamics
Tile id 31 0 0
Tile id 25 0 0
Tile id 30 0 0
Tile id 11 0 0
Tile id 32 0 0
Tile id 26 0 0
Tile id 33 0 0
Tile id 24 0 0
Tile id 20 157 157
Tile id 16 626 626
Tile id 17 733 733
Tile id 69 0 0
Tile id 35 7363 7363
Tile id 72 0 0
Tile id 10 6091 6091
Tile id 34 4122 4122
Tile id 27 14243 14243
Tile id 53 6686 6686
Tile id 18 26155 26155
Tile id 5 18156 18156
Tile id 14 36000 36000
Tile id 19 36608 36608
Tile id 58 30750 30750
Tile id 15 51389 51389
Tile id 9 63958 63958
Tile id 28 74246 74246
Tile id 68 20802 20802
Tile id 64 21894 21894
Tile id 59 76019 76019
Tile id 63 27883 27883
Tile id 29 99786 99786
Tile id 70 29019 29019
Tile id 65 39854 39854
Tile id 4 126014 126014
Tile id 21 150477 150477
Tile id 22 156315 156315
Tile id 23 169974 169974
Tile id 71 79166 79166
Tile id 73 58799 58799
Tile id 52 208843 208843
Tile id 67 141690 141690
Tile id 13 31

... (4 outputs left out)

Eroding (GPU): 100%|██████████| 928/928 [00:06<00:00, 137.50inst/s] 
2026-02-04 14:44:59,965 epicycle.pipeline INFO Stage 'reconstruction': 4/4 succeeded (1477.8s)
2026-02-04 14:44:59,966 epicycle.reconstruction INFO Stage 'reconstruction' finished: 4/4 succeeded (1477.8s)
2026-02-04 14:44:59,967 epicycle.pipeline INFO === Stage: assembly ===
2026-02-04 14:44:59,967 epicycle.assembly INFO Stage 'assembly' starting with 4 tile(s)
2026-02-04 14:44:59,968 epicycle.assembly INFO Parameters: {"full_size": [1792, 1792], "tile_size": 1024, "overlap": 0.25, "depth": null, "stitch_mask": true, "stitch_prob": true, "stitch_binary": true, "prob_stitch_mode": "max", "binary_format": "npy", "n_workers": 16}
2026-02-04 14:44:59,968 epicycle.assembly INFO Loading tiling config from data/WT_Developing_Fish/ScaleCondensate/tiles/tiling_config.json
2026-02-04 14:44:59,969 epicycle.assembly INFO Loading tiling config from data/WT_Developing_Fish/ScaleCondensate/tiles/tiling_config.json
2026-02-04 14:44:5

[ 611 1088 1088] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  252
start dynamics
Tile id 35 0 0
Tile id 23 0 0
Tile id 33 0 0
Tile id 32 0 0
Tile id 5 0 0
Tile id 29 0 0
Tile id 34 0 0
Tile id 17 0 0
Tile id 30 0 0
Tile id 11 0 0
Tile id 31 0 0
Tile id 59 47 47
Tile id 21 0 0
Tile id 20 0 0
Tile id 4 0 0
Tile id 28 0 0
Tile id 24 0 0
Tile id 62 0 0
Tile id 27 0 0
Tile id 16 0 0
Tile id 25 0 0
Tile id 10 0 0
Tile id 61 0 0
Tile id 22 0 0
Tile id 26 0 0
Tile id 60 0 0
Tile id 71 0 0
Tile id 55 51 51
Tile id 47 271 271
Tile id 66 0 0
Tile id 53 474 474
Tile id 69 0 0
Tile id 70 0 0
Tile id 67 0 0
Tile id 65 0 0
Tile id 68 0 0
Tile id 63 0 0
Tile id 56 199 199
Tile id 19 484 484
Tile id 58 424 424
Tile id 64 0 0
Tile id 95 0 0
Tile id 18 689 689
Tile id 101 0 0
Tile id 102 0 0
Tile id 103 0 0
Tile id 54 1780 1780
Tile id 96 0 0
Tile id 98 0 0
Tile id 97 0 0
Tile id 94 133 133
Tile id 107 0 0
Tile id 99 0 0
Tile id 104 0 0
Tile id 3 2226 2226
Tile id 100 0 0
Tile id 41 3435

2026-02-04 14:54:53,132 epicycle.core.global_reconstruction INFO Global reconstruction complete: shape=(515, 896, 896), n_labels=2432, elapsed=492.6s
2026-02-04 14:54:53,133 epicycle.core.global_reconstruction INFO Saving outputs to <run>/WT_Developing_Fish/ScaleCondensate/ScaleCondensate/stage4_global
2026-02-04 14:55:36,997 epicycle.pipeline INFO Stage 'global_reconstruction': 1/1 succeeded (550.5s)
2026-02-04 14:55:36,998 epicycle.global_reconstruction INFO Stage 'global_reconstruction' finished: 1/1 succeeded (550.5s)
2026-02-04 14:55:36,999 epicycle.pipeline WARNING No tiles remaining — stopping pipeline.
2026-02-04 14:55:36,999 epicycle.pipeline INFO Pipeline: 1/4 tiles (2967.4s total)
  tiling: 4/4 succeeded (15.1s)
  segmentation: 4/4 succeeded (837.5s)
  reconstruction: 4/4 succeeded (1477.8s)
  assembly: 4/4 succeeded (86.5s)
  global_reconstruction: 1/1 succeeded (550.5s)



Pipeline: 1/4 tiles (2967.4s total)
  tiling: 4/4 succeeded (15.1s)
  segmentation: 4/4 succeeded (837.5s)
  reconstruction: 4/4 succeeded (1477.8s)
  assembly: 4/4 succeeded (86.5s)
  global_reconstruction: 1/1 succeeded (550.5s)


## 10. Check Status

In [ ]:
# Quick status check — how many tiles completed each stage?
seg_root = output_dir / "stage1_segmentation"
recon_root = output_dir / "stage2_reconstruction"

seg_done = sum(1 for td in tile_dirs if (seg_root / Path(td).name / "SEGMENTATION_DONE").exists())
recon_done = sum(1 for td in tile_dirs if (recon_root / Path(td).name / f"{Path(td).name}_mask.tif").exists())

print(f"Stage 1 (Segmentation):   {seg_done}/{len(tile_dirs)} complete")
print(f"Stage 2 (Reconstruction): {recon_done}/{len(tile_dirs)} complete")
print(f"Stage 3 (Assembly):       check {output_dir / 'stage3_assembly'}")
print(f"Stage 4 (Global Recon):   {'done' if (output_dir / 'stage4_global' / 'global_segmentation.tif').exists() else 'not done'}")

Stage 1 (Segmentation):   0/1 complete
Stage 2 (Reconstruction): 0/1 complete
Stage 3 (Assembly):       check <run>/WT_Developing_Fish/ScaleCondensate/ScaleCondensate/stage3_assembly
Stage 4 (Global Recon):   done
